<a href="https://colab.research.google.com/github/cankuzu/labor-sim/blob/main/notebooks/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# labor-sim on Colab / Kaggle

Runs the experiments on a free T4 GPU with Ollama.

1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells in order. Sessions end after at most 12 hours, so results are saved to Google Drive after every quarter.

In [ ]:
!nvidia-smi

## 1. Install and start Ollama

In [1]:
!git clone -q https://github.com/cankuzu/labor-sim.git /content/labor-sim 2>/dev/null; cd /content/labor-sim && git pull -q && bash scripts/colab_bootstrap.sh

== GPU
Tesla T4, 15360 MiB
== Ollama
{"version":"0.40.18"}
pull llama3.1:8b-instruct-q4_K_M
  ok
NAME                           ID              SIZE      MODIFIED               
llama3.1:8b-instruct-q4_K_M    46e0c10c039e    4.9 GB    Less than a second ago    
== Python packages and data

Wrote 831 occupations to /content/labor-sim/data/processed/occupations.csv
Exposure coverage: {'exposure_eloundou': 0.929, 'exposure_eloundou_human': 0.929, 'exposure_microsoft': 0.943, 'exposure_aioe': 0.929, 'exposure_anthropic': 0.91}
Note: OEWS 2022-2025 not found yet; using [2021] (dev only).
== ready


In [3]:
!cd /content/labor-sim && python scripts/speed_test.py --model llama-instruct --calls 40

{
  "model": "llama-instruct",
  "backend": "ollama",
  "calls": 40,
  "sec_per_call": 2.622,
  "sec_per_call_p90": 2.936,
  "hours_per_run_by_workers": {
    "100": 1.8,
    "250": 3.22,
    "500": 5.59,
    "1000": 10.32
  }
}


In [ ]:
!cd /content/labor-sim && git pull -q && python scripts/run_simulation.py --policy llm --model llama-instruct --workers 100 --seeds 1

  2022Q4: employment 0.950
  2023Q1: employment 0.949


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
server = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434/api/version

## 2. Pull the models
Check the tags in `config/default.yaml` first. Base models without an Ollama tag must be imported from a GGUF file (see README).

In [ ]:
MODELS = ['llama3.1:8b-instruct-q4_K_M', 'qwen2.5:7b-instruct-q4_K_M']  # add the base tags once confirmed
for m in MODELS:
    !ollama pull {m}
!ollama list

## 3. Get the code and save results to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/labor-sim-results'

!git clone https://github.com/cankuzu/labor-sim.git /content/labor-sim 2>/dev/null || git -C /content/labor-sim pull
%cd /content/labor-sim
!pip install -q -r requirements.txt

In [ ]:
open('config/local.yaml', 'w').write(f'''
llm:
  backend: ollama
  ollama_url: http://localhost:11434
paths:
  results: {RESULTS}
''')
!python scripts/download_data.py --only github crosswalk anthropic
!python scripts/build_occupations.py

## 4. Speed test (decides how big the full runs can be)

In [ ]:
import sys, time; sys.path.insert(0, '.')
from labor_sim.config import load_config
from labor_sim.llm.client import make_client
cfg = load_config()
c = make_client('llama-instruct', cfg)
t = time.time()
for i in range(20):
    c.generate('Reply only with JSON: {"action": "search", "reason": "<one sentence>"}', seed=i)
per_call = (time.time() - t) / 20
print(f'{per_call:.2f} s per call')
for workers in (100, 500, 1000):
    calls = (workers + 90) * 13
    print(f'{workers} workers: ~{calls} calls, ~{calls * per_call / 3600:.1f} h per run')

## 5. Pilot runs

In [ ]:
!python scripts/run_survey.py --models llama-instruct --synthetic --n 20 --seeds 1
!python scripts/run_simulation.py --policy llm --model llama-instruct --workers 100 --seeds 1